# Differential Expression Analysis

Author: Franziska Niemeyer

In [ ]:
OUT_DEGS    = "deseq2_results"
PADJ        = 0.05
LFC         = 1.0
SAVE_FIGS   = True
FIG_DIR     = "figures"
FIG_FORMAT  = "png"
FIG_DPI     = 600
PFI_PALETTE = {'short': '#C7844A', 'medium': '#456EAE', 'long': '#538984'}

import os
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import colorsys
from adjustText import adjust_text

In [ ]:
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "legend.fontsize":    7,
    "legend.title_fontsize": 8,
    "figure.titlesize":   13,
    "figure.titleweight": "bold",
    "figure.dpi":         300,
})

## Load results

In [ ]:
def get_sig(results_df, padj=PADJ, lfc=LFC):
    """Filter DESeq2 results to significant DEGs."""
    return results_df[
        (results_df["padj"] < padj) &
        (results_df["log2FoldChange"].abs() > lfc)
    ].sort_values("padj")


def load_contrast(name, output_dir=OUT_DEGS):
    """
    Load full, shrunk and significant CSVs for one contrast.
    Returns a dict with keys 'full', 'shrunk', 'significant'.
    Falls back to 'full' for the shrunk key if no _shrunk.csv exists
    (e.g. if the run used shrink_coeff=None).
    """
    if name.startswith("condition_condition_in_"):
        name = name.replace("condition_condition_in_", "condition_in_class_")
    name_clean = name.replace(" ", "_").replace(":", "_")
    full_path = os.path.join(output_dir, f"{name_clean}_full.csv")
    sig_path = os.path.join(output_dir, f"{name_clean}_significant.csv")
    shrunk_path = os.path.join(output_dir, f"{name_clean}_shrunk.csv")
                
    if not os.path.exists(full_path):
        print(f"  WARNING: {full_path} not found — skipping")
        return None

    result = {
        "full":        pd.read_csv(full_path, index_col=0),
        "significant": pd.read_csv(sig_path,  index_col=0),
    }
    if os.path.exists(shrunk_path):
        result["shrunk"] = pd.read_csv(shrunk_path, index_col=0)
    else:
        # No separate shrunk file — use full (unshrunk) LFCs for plotting
        result["shrunk"] = result["full"]

    return result


summary_path = os.path.join(OUT_DEGS, "summary.csv")
summary_df   = pd.read_csv(summary_path)

print(f"Found {len(summary_df)} contrasts:\n")
print(summary_df.to_string(index=False))

In [ ]:
# ── Load all contrasts ────────────────────────────────────────────────────────
results = {}
for name in summary_df["comparison"]:
    loaded = load_contrast(name)
    if loaded is not None:
        results[name] = loaded

print(f"\nLoaded {len(results)} contrasts successfully.")

In [ ]:
# Condition overall
res_condition = results["condition_short_vs_long"]["full"]
res_condition_shrunk = results["condition_short_vs_long"]["shrunk"]

# Histology pairwise contrasts
res_histology = {
    name.replace("histology_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("histology_")
}
res_histology_shrunk = {
    name.replace("histology_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("histology_")
}

# Class pairwise contrasts
res_class = {
    name.replace("class_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("class_")
}
res_class_shrunk = {
    name.replace("class_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("class_")
}

# Condition within each class (saved as condition_condition_in_{cls})
res_condition_in_class = {
    name.replace("condition_condition_in_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("condition_condition_in_")
}
res_condition_in_class_shrunk = {
    name.replace("condition_condition_in_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("condition_condition_in_")
}

# Condition within each histology (saved as condition_in_histology_{hist})
res_condition_in_histology = {
    name.replace("condition_in_histology_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("condition_in_histology_")
}
res_condition_in_histology_shrunk = {
    name.replace("condition_in_histology_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("condition_in_histology_")
}

print("res_condition               :", "loaded" if res_condition is not None else "NOT FOUND")
print("res_histology               :", list(res_histology.keys()))
print("res_class                   :", list(res_class.keys()))
print("res_condition_in_class      :", list(res_condition_in_class.keys()))
print("res_condition_in_histology  :", list(res_condition_in_histology.keys()))

In [ ]:
print("Sanity check — significant DEG counts vs summary table:")
for name, data in results.items():
    n_sig_recomputed = len(get_sig(data["full"]))
    n_sig_saved      = summary_df.loc[
        summary_df["comparison"] == name, "n_significant"
    ].values[0]
    match = "✓" if n_sig_recomputed == n_sig_saved else "✗ MISMATCH"
    print(f"  {match}  {name}: {n_sig_recomputed} (recomputed) vs {n_sig_saved} (saved)")


## Summary: DEG counts across all contrasts

In [ ]:
def plot_summary_bar(summary_df, padj=PADJ, lfc=LFC, figsize=(10, 5)):
    """
    Horizontal stacked bar chart showing up- and down-regulated DEG counts
    for every contrast, sorted by total significant genes.
    """
    df = summary_df.sort_values("n_significant", ascending=True).copy()

    fig, ax = plt.subplots(figsize=figsize)

    y = np.arange(len(df))
    ax.barh(y,  df["n_upregulated"],   color="#C0392B", label="Up-regulated",   height=0.6)
    ax.barh(y, -df["n_downregulated"], color="#2980B9", label="Down-regulated", height=0.6)

    # Labels showing total significant per contrast
    for i, (_, row) in enumerate(df.iterrows()):
        total = row["n_significant"]
        if total > 0:
            ax.text(row["n_upregulated"] + 2, i, str(int(total)),
                    va="center", fontsize=8, color="#333333")

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(df["comparison"], fontsize=9)
    ax.set_xlabel("Number of significant DEGs", fontsize=11)
    ax.set_title(f"Significant DEGs per contrast  (padj<{padj}, |LFC|>{lfc})",
                 fontsize=12, pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=10, loc="lower right")

    # Symmetric x-axis
    xmax = max(df["n_upregulated"].max(), df["n_downregulated"].max()) * 1.15
    ax.set_xlim(-xmax, xmax)

    # Replace negative x tick labels with positive values
    xticks = ax.get_xticks()
    ax.set_xticklabels([str(int(abs(t))) for t in xticks], fontsize=9)

    plt.tight_layout()
    return fig, ax


fig, ax = plot_summary_bar(summary_df, figsize=(10,4))
if SAVE_FIGS:
    fig.savefig(os.path.join(FIG_DIR, f"summary_DEG_counts.{FIG_FORMAT}"),
                dpi=FIG_DPI, bbox_inches="tight")
plt.show()

## MA plots

In [ ]:
def plot_ma(results_df, title="MA plot",
            padj=PADJ, lfc=LFC,
            genes_of_interest=None,
            label_fontsize=8,
            figsize=(6, 5)):
    """
    MA plot: mean expression (x, log10) vs log2FC (y).

    Significant genes (padj < threshold AND |LFC| > threshold) are
    highlighted in purple; all others in yellow. Genes of interest
    are annotated with labels.

    Parameters
    ----------
    results_df        : pd.DataFrame  full DESeq2 results table
    title             : str
    padj, lfc         : float         significance thresholds
    genes_of_interest : list[str]     genes to annotate
    """
    genes_of_interest = genes_of_interest or []

    df = results_df.dropna(subset=["baseMean", "log2FoldChange", "padj"]).copy()
    df = df[df["baseMean"] > 0]

    sig_mask = (df["padj"] < padj) & (df["log2FoldChange"].abs() > lfc)

    fig, ax = plt.subplots(figsize=figsize)

    # Non-significant
    ax.scatter(df.loc[~sig_mask, "baseMean"],
               df.loc[~sig_mask, "log2FoldChange"],
               s=6, c="#F1C40F", alpha=0.4, edgecolors="none",
               rasterized=True, zorder=1, label="Not significant")

    # Significant
    ax.scatter(df.loc[sig_mask, "baseMean"],
               df.loc[sig_mask, "log2FoldChange"],
               s=12, c="#6C3483", alpha=0.8, edgecolors="none",
               rasterized=True, zorder=2,
               label=f"Significant (padj<{padj}, |LFC|>{lfc})")

    ax.axhline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_xscale("log")
    ax.set_xlabel("Mean expression", fontsize=11)
    ax.set_ylabel("log2FC", fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=True, fontsize=9, loc="upper right")

    # Annotate genes of interest
    texts = []
    goi_present = [g for g in genes_of_interest if g in df.index]
    for gene in goi_present:
        row = df.loc[gene]
        ax.scatter(row["baseMean"], row["log2FoldChange"],
                   s=40, c="#E74C3C", edgecolors="black",
                   linewidths=0.8, zorder=4)
        txt = ax.text(row["baseMean"], row["log2FoldChange"], gene,
                      fontsize=label_fontsize, fontweight="bold", zorder=5)
        txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
        texts.append(txt)

    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle="-", lw=0.5, color="black"))

    plt.tight_layout()
    return fig, ax

In [ ]:
# ── MA plots for all contrasts ────────────────────────────────────────────────
GENES_OF_INTEREST = ["C3", "IFI27", "BST2"]

for contrast_name, data in results.items():
    # Use shrunk LFCs for the y-axis if available
    plot_df = data["shrunk"] if "shrunk" in data else data["full"]
    # baseMean comes from the unshrunk full results
    if "baseMean" not in plot_df.columns:
        plot_df = plot_df.copy()
        plot_df["baseMean"] = data["full"]["baseMean"]
    # Always use unshrunk padj for significance calls
    plot_df = plot_df.copy()
    plot_df["padj"] = data["full"]["padj"]

    fig, ax = plot_ma(
        plot_df,
        title=f"MA plot — {contrast_name.replace('_', ' ')}",
        genes_of_interest=GENES_OF_INTEREST,
    )
    if SAVE_FIGS:
        fig.savefig(
            os.path.join(FIG_DIR, f"MA_{contrast_name}.{FIG_FORMAT}"),
            dpi=FIG_DPI, bbox_inches="tight"
        )
    plt.show()

## Volcano plots

In [ ]:
def plot_volcano(results_df, title="Volcano plot",
                 padj=PADJ, lfc=LFC,
                 genes_of_interest=None,
                 label_fontsize=8,
                 figsize=(6, 5),
                 max_neg_log10_padj=50):
    """
    Volcano plot: log2FC (x) vs -log10(padj) (y).

    Points are coloured by direction of significant change.
    Genes of interest are annotated regardless of significance.

    Parameters
    ----------
    max_neg_log10_padj : float  cap -log10(padj) to avoid extreme y values
                                from genes with padj ~ 0
    """
    genes_of_interest = genes_of_interest or []

    df = results_df.dropna(subset=["log2FoldChange", "padj"]).copy()
    # Cap extreme p-values for visual clarity
    df["neg_log10_padj"] = np.minimum(
        -np.log10(df["padj"].clip(lower=1e-300)),
        max_neg_log10_padj
    )

    # Classify each gene
    up_mask   = (df["padj"] < padj) & (df["log2FoldChange"] >  lfc)
    down_mask = (df["padj"] < padj) & (df["log2FoldChange"] < -lfc)
    ns_mask   = ~(up_mask | down_mask)

    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(df.loc[ns_mask,   "log2FoldChange"],
               df.loc[ns_mask,   "neg_log10_padj"],
               s=6, c="#BDC3C7", alpha=0.4, edgecolors="none",
               rasterized=True, zorder=1, label="Not significant")
    ax.scatter(df.loc[up_mask,   "log2FoldChange"],
               df.loc[up_mask,   "neg_log10_padj"],
               s=8, c="#C0392B", alpha=0.75, edgecolors="none",
               rasterized=True, zorder=2,
               label=f"Up (LFC>{lfc}, padj<{padj})  n={up_mask.sum()}")
    ax.scatter(df.loc[down_mask, "log2FoldChange"],
               df.loc[down_mask, "neg_log10_padj"],
               s=8, c="#2980B9", alpha=0.75, edgecolors="none",
               rasterized=True, zorder=2,
               label=f"Down (LFC<-{lfc}, padj<{padj})  n={down_mask.sum()}")

    # Threshold lines
    ax.axhline(-np.log10(padj), color="grey", linestyle="--",
               linewidth=0.8, zorder=0)
    ax.axvline( lfc, color="grey", linestyle="--", linewidth=0.8, zorder=0)
    ax.axvline(-lfc, color="grey", linestyle="--", linewidth=0.8, zorder=0)

    ax.set_xlabel("log2 Fold Change", fontsize=11)
    ax.set_ylabel("-log10(padj)", fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left")

    # Annotate genes of interest
    texts = []
    goi_present = [g for g in genes_of_interest if g in df.index]
    for gene in goi_present:
        row = df.loc[gene]
        ax.scatter(row["log2FoldChange"], row["neg_log10_padj"],
                   s=40, c="#E67E22", edgecolors="black",
                   linewidths=0.8, zorder=4)
        txt = ax.text(row["log2FoldChange"], row["neg_log10_padj"],
                      gene, fontsize=label_fontsize,
                      fontweight="bold", zorder=5)
        txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
        texts.append(txt)

    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle="-", lw=0.5, color="black"))

    plt.tight_layout()
    return fig, ax

In [ ]:
# ── Volcano plots for all contrasts ──────────────────────────────────────────
for contrast_name, data in results.items():
    # Use shrunk LFCs for x-axis positions, unshrunk padj for y-axis
    plot_df = data["shrunk"].copy()
    plot_df["padj"] = data["full"]["padj"]

    title = contrast_name.replace('_', ' ')
    if title.startswith("condition condition"):
        title = title.replace("condition condition", "condition")

    fig, ax = plot_volcano(
        plot_df,
        title=f"DEGs — {title}",
        genes_of_interest=GENES_OF_INTEREST,
    )
    if SAVE_FIGS:
        fig.savefig(
            os.path.join(FIG_DIR, f"volcano_{contrast_name}.{FIG_FORMAT}"),
            dpi=FIG_DPI, bbox_inches="tight"
        )
    plt.show()

## Effect comparison scatter plots

In [ ]:
def plot_effect_comparison(
    res_df_x,
    res_df_y,
    genes_of_interest=None,
    padj_thresh=PADJ,
    lfc_thresh=None,
    title="Effect comparison",
    xlabel="X log2FC",
    ylabel="Y log2FC",
    x_label="X",
    y_label="Y",
    figsize=(6.5, 6.5),
    annotate_all_goi=True,
    annotate_only_sig_goi=False,
    label_fontsize=9,
    point_size=14,
    goi_point_size=50,
    n_top_specific=5,
):
    """
    Scatter plot comparing log2FCs between two contrasts.
    """
    genes_of_interest = genes_of_interest or []

    # ── Merge ─────────────────────────────────────────────────────────────────
    merged = (
        res_df_x[["log2FoldChange", "padj"]]
        .rename(columns={"log2FoldChange": "lfc_x", "padj": "padj_x"})
        .join(
            res_df_y[["log2FoldChange", "padj"]]
            .rename(columns={"log2FoldChange": "lfc_y", "padj": "padj_y"}),
            how="inner",
        )
    )
    n_dropped = len(res_df_x) + len(res_df_y) - 2 * len(merged)
    if n_dropped > 0:
        print(f"  Note: {n_dropped} genes dropped (not present in both contrasts)")

    # ── Significance classification ───────────────────────────────────────────
    merged["sig_x"] = merged["padj_x"] < padj_thresh
    merged["sig_y"] = merged["padj_y"] < padj_thresh
    if lfc_thresh is not None:
        merged["sig_x"] &= merged["lfc_x"].abs() >= lfc_thresh
        merged["sig_y"] &= merged["lfc_y"].abs() >= lfc_thresh

    def classify(row):
        if row["sig_x"] and row["sig_y"]:   return "Both"
        elif row["sig_x"]:                  return f"{x_label} only"
        elif row["sig_y"]:                  return f"{y_label} only"
        else:                               return "Not significant"

    merged["group"] = merged.apply(classify, axis=1)

    color_map = {
        "Not significant":  "#D9D9D9",
        f"{x_label} only":  "#4C78A8",
        f"{y_label} only":  "#F58518",
        "Both":             "#B22222",
    }

    # ── Most group-specific genes ─────────────────────────────────────────────
    # X only: significant in x but not y — rank by |lfc_x|
    top_x_only = (
        merged[merged["group"] == f"{x_label} only"]
        .assign(rank_metric=lambda d: d["lfc_x"].abs())
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )
    # Y only: significant in y but not x — rank by |lfc_y|
    top_y_only = (
        merged[merged["group"] == f"{y_label} only"]
        .assign(rank_metric=lambda d: d["lfc_y"].abs())
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )
    # Both significant, more extreme in x direction (above diagonal)
    top_both_x = (
        merged[merged["group"] == "Both"]
        .assign(rank_metric=lambda d: d["lfc_x"] - d["lfc_y"])
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )
    # Both significant, more extreme in y direction (below diagonal)
    top_both_y = (
        merged[merged["group"] == "Both"]
        .assign(rank_metric=lambda d: d["lfc_y"] - d["lfc_x"])
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )

    # Combine all specific genes
    goi_set       = set(genes_of_interest)
    specific_genes = {
        gene: group
        for group, genes in [
            (f"{x_label} only", top_x_only),
            (f"{y_label} only", top_y_only),
            (f"Both → {x_label}", top_both_x),
            (f"Both → {y_label}", top_both_y),
        ]
        for gene in genes
        if gene not in goi_set
    }

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    for group in ["Not significant", f"{x_label} only", f"{y_label} only", "Both"]:
        sub = merged[merged["group"] == group]
        ax.scatter(
            sub["lfc_x"], sub["lfc_y"],
            s=point_size,
            c=color_map[group],
            alpha=0.75 if group != "Not significant" else 0.45,
            edgecolors="none",
            rasterized=True,
            zorder=2 if group != "Not significant" else 1,
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=0.9, zorder=0)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.9, zorder=0)

    finite_vals = np.r_[merged["lfc_x"].values, merged["lfc_y"].values]
    finite_vals = finite_vals[np.isfinite(finite_vals)]
    lim = np.nanmax(np.abs(finite_vals))
    lim = max(lim, 1.0)
    lim = np.ceil(lim * 1.05 * 2) / 2
    ax.plot([-lim, lim], [-lim, lim],
            linestyle=":", linewidth=1.2, color="#666666", zorder=0)

    # ── Annotate specific genes ───────────────────────────────────────────────
    specific_color_map = {
        f"{x_label} only":   color_map[f"{x_label} only"],
        f"{y_label} only":   color_map[f"{y_label} only"],
        f"Both → {x_label}": "#8B0000",   # dark red, x-biased
        f"Both → {y_label}": "#8B4500",   # dark orange, y-biased
    }

    texts = []
    for gene, group in specific_genes.items():
        if gene not in merged.index:
            continue
        row   = merged.loc[gene]
        color = specific_color_map[group]
        # Open circle with coloured edge
        ax.scatter(
            row["lfc_x"], row["lfc_y"],
            s=goi_point_size * 0.8,
            facecolors="none",
            edgecolors=color,
            linewidths=1.5,
            zorder=4,
        )
        txt = ax.text(
            row["lfc_x"], row["lfc_y"], gene,
            fontsize=label_fontsize - 1,
            color=color,
            fontstyle="italic",   # italic to distinguish from GOI bold labels
            zorder=5,
        )
        txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
        texts.append(txt)

    # ── Annotate genes of interest ────────────────────────────────────────────
    goi_present = [g for g in genes_of_interest if g in merged.index]
    goi_missing = [g for g in genes_of_interest if g not in merged.index]
    if goi_missing:
        print(f"  GOI not in merged results: {goi_missing}")

    if goi_present:
        goi_df     = merged.loc[goi_present].copy()
        if annotate_only_sig_goi:
            goi_df = goi_df[goi_df["sig_x"] | goi_df["sig_y"]]
        goi_colors = goi_df["group"].map(
            lambda g: darken_color(color_map.get(g, "#888888"), 0.8)
        )
        ax.scatter(
            goi_df["lfc_x"], goi_df["lfc_y"],
            s=goi_point_size,
            c=goi_colors,
            edgecolors="black",
            linewidths=0.9,
            zorder=5,
        )
        for gene, row in goi_df.iterrows():
            txt = ax.text(
                row["lfc_x"], row["lfc_y"], gene,
                fontsize=label_fontsize,
                color="black",
                fontweight="bold",   # bold to distinguish from specific genes
                zorder=6,
            )
            txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])
            texts.append(txt)

    adjust_text(
        texts, ax=ax,
        expand=(1.15, 1.25),
        arrowprops=dict(arrowstyle="-", lw=0.6, color="#888888",
                       shrinkA=3, shrinkB=3),
    )

    # ── Styling ───────────────────────────────────────────────────────────────
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=11, pad=15, loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=10, width=0.9)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_elements = [
        Line2D([0],[0], marker="o", color="w", label="Not significant",
               markerfacecolor=color_map["Not significant"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label=f"{x_label} only",
               markerfacecolor=color_map[f"{x_label} only"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label=f"{y_label} only",
               markerfacecolor=color_map[f"{y_label} only"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label="Both",
               markerfacecolor=color_map["Both"], markersize=6),
        # Specific gene markers — open circles
        Line2D([0],[0], marker="o", color="w",
               label=f"Top specific to {x_label}",
               markerfacecolor="none",
               markeredgecolor=color_map[f"{x_label} only"],
               markeredgewidth=1.5, markersize=7),
        Line2D([0],[0], marker="o", color="w",
               label=f"Top specific to {y_label}",
               markerfacecolor="none",
               markeredgecolor=color_map[f"{y_label} only"],
               markeredgewidth=1.5, markersize=7),
        Line2D([0],[0], marker="o", color="w",
               label=f"Both, biased to {x_label}",
               markerfacecolor="none",
               markeredgecolor="#8B0000",
               markeredgewidth=1.5, markersize=7),
        Line2D([0],[0], marker="o", color="w",
               label=f"Both, biased to {y_label}",
               markerfacecolor="none",
               markeredgecolor="#8B4500",
               markeredgewidth=1.5, markersize=7),
    ]
    if goi_present:
        legend_elements.append(
            Line2D([0],[0], marker="o", color="w", label="Gene of interest",
                   markerfacecolor="#C00000", markeredgecolor="black", markersize=7)
        )
    ax.legend(
        handles=legend_elements,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.03),
        borderaxespad=0,
        fontsize=9,
    )

    plt.tight_layout()
    return fig, ax, merged

## Top DEG tables

In [ ]:
# ── Display top significant DEGs for each contrast ────────────────────────────
N_TOP = 20   # number of top genes to display per contrast

for contrast_name, data in results.items():
    sig = get_sig(data["full"])
    if len(sig) == 0:
        print(f"\n{contrast_name}: no significant DEGs")
        continue

    print(f"\n{'─'*60}")
    print(f"{contrast_name}  —  {len(sig)} significant DEGs  "
          f"(showing top {min(N_TOP, len(sig))})")
    print(f"{'─'*60}")

    display_cols = [c for c in
                    ["baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj"]
                    if c in sig.columns]
    display(
        sig[display_cols]
        .head(N_TOP)
        .style
        .format({
            "baseMean":        "{:.1f}",
            "log2FoldChange":  "{:.3f}",
            "lfcSE":           "{:.3f}",
            "stat":            "{:.2f}",
            "pvalue":          "{:.2e}",
            "padj":            "{:.2e}",
        })
        .background_gradient(subset=["log2FoldChange"], cmap="RdBu_r",
                             vmin=-3, vmax=3)
        .background_gradient(subset=["padj"], cmap="YlOrRd_r",
                             vmin=0, vmax=PADJ)
    )

## Export top DEG gene lists

In [ ]:
# ── Write a simple gene list (one gene per line) for each contrast ────────────
# Useful as input to GSEA pre-ranked or pathway enrichment tools
GENELIST_DIR = os.path.join(OUT_DEGS, "gene_lists")
os.makedirs(GENELIST_DIR, exist_ok=True)

for contrast_name, data in results.items():
    sig = get_sig(data["full"])
    if len(sig) == 0:
        continue

    # Ranked by padj ascending (most significant first)
    gene_list = sig.index.tolist()
    out_path  = os.path.join(GENELIST_DIR, f"{contrast_name}_sig_genes.txt")
    with open(out_path, "w") as f:
        f.write("\n".join(gene_list))
    print(f"  {contrast_name}: {len(gene_list)} genes → {out_path}")

    # Also write a ranked list with LFC for pre-ranked GSEA
    # Format: gene <tab> ranking_metric (signed -log10 padj * sign of LFC)
    rnk_df  = data["full"].copy()
    rnk_df  = rnk_df.dropna(subset=["log2FoldChange", "padj"])
    rnk_df["rank_metric"] = (
        np.sign(rnk_df["log2FoldChange"]) *
        -np.log10(rnk_df["padj"].clip(lower=1e-300))
    )
    rnk_df = rnk_df.sort_values("rank_metric", ascending=False)
    rnk_path = os.path.join(GENELIST_DIR, f"{contrast_name}_ranked.rnk")
    rnk_df[["rank_metric"]].to_csv(rnk_path, sep="\t", header=False)
    print(f"  {contrast_name}: ranked list → {rnk_path}")

print("\nDone.")

In [ ]:
# ── Load pseudobulk counts and metadata ───────────────────────────────────────
counts_pb  = pd.read_csv(os.path.join(OUT_DEGS, "pseudobulk_counts.csv"),
                         index_col=0)
metadata_pb = pd.read_csv(os.path.join(OUT_DEGS, "pseudobulk_metadata.csv"),
                          index_col=0)

# Restore categorical dtypes lost during CSV round-trip
for col in ["condition", "histology", "class", "patient"]:
    if col in metadata_pb.columns:
        metadata_pb[col] = metadata_pb[col].astype("category")

print(f"Pseudobulk matrix : {counts_pb.shape[0]} samples × {counts_pb.shape[1]} genes")
print(metadata_pb.head())

### Heatmap

In [ ]:
# ── Load pseudobulk counts and metadata ───────────────────────────────────────
counts_pb  = pd.read_csv(os.path.join(OUT_DEGS, "pseudobulk_counts.csv"),
                         index_col=0)
metadata_pb = pd.read_csv(os.path.join(OUT_DEGS, "pseudobulk_metadata.csv"),
                          index_col=0)

for col in ["condition", "histology", "patient"]:
    if col in metadata_pb.columns:
        metadata_pb[col] = metadata_pb[col].astype("category")

print(f"Pseudobulk matrix : {counts_pb.shape[0]} samples × {counts_pb.shape[1]} genes")
print(metadata_pb.head())

In [ ]:
from matplotlib.patches import Patch
import seaborn as sns

CONTRAST_NAME     = "condition_short_vs_long"
N_TOP_GENES       = 30
GENES_OF_INTEREST = []

condition_palette = {
    "long":  "#7A7A7A",
    "short": "#C44E52",
}
condition_palette = {k: PFI_PALETTE[k] for k in ["short", "long"]} if PFI_PALETTE else condition_palette

hist_palette = {
    "Tumor epithelium":   "#C0392B",
    "Stroma":             "#E8A020",
    "Tumor stroma":       "#C97D4E",
}

In [ ]:
metadata_pb['histology'].value_counts()

In [ ]:
def plot_deg_clustermap(
    results_df,
    counts_pb,
    metadata_pb,
    contrast_name,
    n_top=N_TOP_GENES,
    genes_of_interest=None,
    condition_palette=condition_palette,
    class_palette=class_palette,
    hist_palette=hist_palette,
    figsize=(10, 4),
    fig_title=None,
    save_path=None,
):
    genes_of_interest = genes_of_interest or []

    # ── Select genes to plot ──────────────────────────────────────────────────
    top_genes  = (
        results_df
        .dropna(subset=["padj"])
        .sort_values("padj")
        .head(n_top)
        .index.tolist()
    )
    goi_present = [g for g in genes_of_interest if g in counts_pb.columns]
    top_only    = [g for g in top_genes if g not in goi_present]
    genes_plot  = [g for g in goi_present + top_only if g in counts_pb.columns]

    if not genes_plot:
        print("No genes to plot — check gene names match between "
              "results_df and counts_pb")
        return None, None

    # ── Build z-scored log1p matrix at sample level first ────────────────────
    shared_idx = counts_pb.index.intersection(metadata_pb.index)
    mat = np.log1p(counts_pb.loc[shared_idx, genes_plot])
    mat = (mat - mat.mean(axis=0)) / mat.std(axis=0)

    # ── Average across patients within each condition × class × histology ─────
    # This collapses patient-level replication so each row in the heatmap
    # represents one biological group rather than one patient
    meta_sub = metadata_pb.loc[shared_idx].copy()
    mat.index = pd.MultiIndex.from_frame(
        meta_sub[["condition", "histology"]]
    )
    mat = mat.groupby(level=["condition", "histology"]).mean()

    # mat index is now a MultiIndex of (condition, histology)
    # Convert back to a flat DataFrame for seaborn
    mat.index = [
        f"{c} · {h}"
        for c, h in mat.index
    ]

    mat = np.log1p(counts_pb.loc[shared_idx, genes_plot])
    mat = (mat - mat.mean(axis=0)) / mat.std(axis=0)
    
    # Constant genes (std=0) produce NaN after z-scoring → replace with 0
    n_const = mat.isna().any(axis=0).sum()
    if n_const:
        print(f"  {n_const} constant gene(s) set to z=0: "
              f"{mat.columns[mat.isna().any(axis=0)].tolist()}")
    mat = mat.fillna(0)

    # ── Reconstruct group-level metadata for colour bars ──────────────────────
    group_meta = pd.DataFrame(
        mat.index.str.split(" · ").tolist(),
        columns=["condition", "histology"],
        index=mat.index,
    )

    # ── Sort rows by condition → class → histology ────────────────────────────
    group_meta["condition"] = pd.Categorical(
        group_meta["condition"],
        categories=list(condition_palette.keys()),
        ordered=True,
    )
    order      = group_meta.sort_values(["condition", "histology"]).index
    mat        = mat.loc[order]
    group_meta = group_meta.loc[order]

    # ── Row colour bars ───────────────────────────────────────────────────────
    row_colors = pd.DataFrame({
        "outcome":   group_meta["condition"].map(condition_palette),
        "histology": group_meta["histology"].map(hist_palette),
    }, index=mat.index)

    # ── Clustermap ────────────────────────────────────────────────────────────
    g = sns.clustermap(
        mat,
        row_colors=row_colors,
        figsize=figsize,
        cmap="RdBu_r",
        center=0,
        xticklabels=True,
        yticklabels=True,
        row_cluster=False,
    )
    
    hm = g.ax_heatmap
    pos = hm.get_position()
    row_col = g.ax_row_colors
    row_col_pos = row_col.get_position()
    
    gap = 0.005
    hm.set_position([pos.x0 + gap, pos.y0, pos.width - gap, pos.height])

    # Bold genes of interest on x-axis
    plt.gcf().canvas.draw()
    goi_set = set(genes_of_interest)
    for lbl in g.ax_heatmap.get_xticklabels():
        if lbl.get_text() in goi_set:
            lbl.set_fontweight("bold")

    # Clean y-axis labels — already formatted as "condition · histology"
    # but shorten histology names for readability
    def clean_label(lbl):
        return (lbl.get_text()
                .replace("Tumor epithelium", "Tumor")
                .replace("Mixed TumEpi+Other", "Mixed"))
                # .replace("Other", "Stroma"))

    g.ax_heatmap.set_yticklabels(
        [clean_label(lbl) for lbl in g.ax_heatmap.get_yticklabels()],
        rotation=0,
        fontsize=8,
    )
    g.ax_heatmap.set_title(
        label=f"Top {len(genes_plot)} DEGs — {contrast_name.replace('_', ' ')}" if not fig_title else fig_title,
        pad=50,
        fontsize=11
    )

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_handles = (
        [Patch(facecolor=v, edgecolor="none", label=f"outcome: {k}")
         for k, v in condition_palette.items()] +
        [Patch(facecolor=v, edgecolor="none", label=f"class: {k}")
         for k, v in class_palette.items()] +
        [Patch(facecolor=v, edgecolor="none", label=f"histology: {k}")
         for k, v in hist_palette.items()]
    )
    g.ax_heatmap.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(-0.6, 0.02),
        frameon=False,
        fontsize=8,
    )
    g.cax.set_ylabel("z-score", fontsize=8, rotation=90, labelpad=-55) 
    g.cax.set_position([0.1, 0.4, 0.02, 0.4])

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()
    return g, mat

In [ ]:
g, mat = plot_deg_clustermap(
    results_df=results[CONTRAST_NAME]["full"],
    counts_pb=counts_pb,
    metadata_pb=metadata_pb,
    contrast_name=CONTRAST_NAME,
    n_top=N_TOP_GENES,
    fig_title="Top 30 DEGs + genes of interest - short vs long & medium PFI",
    genes_of_interest=["C3", "IFI27", "BST2"],
    save_path=os.path.join(FIG_DIR, f"clustermap_{CONTRAST_NAME}.{FIG_FORMAT}"),
)